# ACUG-LightNet: Full Reproducible Implementation (Colab-ready)

**Work:** *Anatomy-Constrained Topology-Aware Foundation Learning and Uncertainty-Guided Lesion Reconstruction for Lightweight CT-Based Stroke Detection*

This notebook provides a runnable research implementation of the core pipeline described in `SS-69.docx`: multi-window CT input, anatomy/attenuation-conditioned feature fusion, joint classification and lesion-map learning, evidential uncertainty, validation-set temperature scaling, uncertainty-triggered refinement, selective review/abstention, retrieval-augmented guideline evidence, metrics, and a quantized-model benchmark.

**Important implementation boundary:** the manuscript refers to specific pretrained 3D encoders (TAP-CT / DINO-Med3D), clinical datasets, and guideline files that are not bundled with the manuscript. This notebook therefore implements a lightweight 2D/2.5D PyTorch analogue that runs with user-provided image/mask data and includes a clearly labelled synthetic smoke-test mode. It does **not** claim to reproduce the manuscript's pretrained 3D backbone or clinical results. Do not use for diagnosis. External evaluation requires separately obtained, properly licensed datasets and patient-level metadata.

## 1. Environment and reproducibility

In [ ]:
# Colab: uncomment if needed
# from google.colab import drive
# drive.mount('/content/drive')

import os, sys, json, math, time, random, warnings, copy, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, brier_score_loss,
    classification_report, matthews_corrcoef)
from sklearn.calibration import calibration_curve
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__, '| Device:', DEVICE)

## 2. Configure dataset and input schema
The easiest supported format is a CSV with columns for image path, patient/group ID, binary label, and optional lesion-mask path. Image and mask paths may be absolute or relative to `DATA_ROOT`. Labels: `0 = no target lesion/negative`, `1 = stroke/hemorrhage-positive` for a single binary experiment. For the manuscript's partial-label setting, use a curated harmonized CSV and set `label_valid` / `mask_valid`; an unobserved label must not be converted into a negative label.

In [ ]:
# EDIT THESE PATHS for your environment. Example:
# DATA_ROOT = '/content/drive/MyDrive/ACUG_dataset'
# CSV_PATH  = f'{DATA_ROOT}/metadata.csv'
DATA_ROOT = '/content/drive/MyDrive/ACUG_dataset'
CSV_PATH = f'{DATA_ROOT}/metadata.csv'

IMAGE_COL = 'image_path'
LABEL_COL = 'label'
PATIENT_COL = 'patient_id'
MASK_COL = 'mask_path'       # optional; if absent, segmentation loss is skipped
LABEL_VALID_COL = 'label_valid' # optional; defaults to 1
MASK_VALID_COL = 'mask_valid'   # optional; defaults to 1 only when a mask path exists
IMAGE_SIZE = 224
BATCH_SIZE = 8
NUM_WORKERS = 2
EPOCHS = 8
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
USE_SYNTHETIC_SMOKE_TEST = True  # set False after configuring a real CSV

## 3. Data loading, multi-window encoding, and visualization
For DICOM/NIfTI volumes, first export aligned axial slices and masks to image files while retaining patient IDs and physical spacing. The helper below accepts grayscale PNG/JPEG/TIFF. It applies three intensity windows as a practical approximation; true HU windows require original calibrated HU values.

In [ ]:
def resolve_path(p, root=DATA_ROOT):
    if p is None or (isinstance(p, float) and np.isnan(p)) or str(p).strip() == '':
        return None
    p = Path(str(p))
    return p if p.is_absolute() else Path(root) / p

def read_gray(path):
    im = Image.open(path).convert('L').resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
    return np.asarray(im, dtype=np.float32)

def window_channels(arr):
    # For exported HU images, replace these with clinically appropriate HU windowing.
    # For ordinary 8-bit PNG, normalize and create complementary contrast channels.
    x = arr / 255.0 if arr.max() > 1.5 else arr
    x = np.clip(x, 0, 1)
    ch1 = x
    ch2 = np.clip((x - 0.20) / 0.55, 0, 1)
    ch3 = np.clip((x - 0.35) / 0.35, 0, 1)
    return np.stack([ch1, ch2, ch3], axis=0).astype(np.float32)

class CT2DDataset(Dataset):
    def __init__(self, frame, training=False):
        self.df = frame.reset_index(drop=True)
        self.training = training
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        ip = resolve_path(r[IMAGE_COL])
        arr = read_gray(ip)
        x = window_channels(arr)
        do_flip = self.training and random.random() < 0.5
        if do_flip:
            x = x[:, :, ::-1].copy()
        mask = np.zeros((IMAGE_SIZE, IMAGE_SIZE), np.float32)
        mask_valid = int(r.get(MASK_VALID_COL, 0)) if MASK_VALID_COL in self.df.columns else 0
        mp = resolve_path(r.get(MASK_COL, None)) if MASK_COL in self.df.columns else None
        if mp is not None and mp.exists() and mask_valid:
            m = Image.open(mp).convert('L').resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.NEAREST)
            mask = (np.asarray(m, dtype=np.uint8) > 0).astype(np.float32)
            if do_flip:
                mask = mask[:, ::-1].copy()
        label_valid = int(r.get(LABEL_VALID_COL, 1)) if LABEL_VALID_COL in self.df.columns else 1
        y = int(r[LABEL_COL]) if label_valid else -1
        return (torch.tensor(x), torch.tensor(y, dtype=torch.long),
                torch.tensor(mask[None]), torch.tensor(float(mask_valid)),
                torch.tensor(float(label_valid)), str(r.get(PATIENT_COL, i)))

def make_synthetic_frame(n=240):
    # Pipeline smoke test only; synthetic images are NOT medical data.
    rng = np.random.default_rng(SEED)
    out = []
    base = Path('/tmp/acug_synthetic')
    (base/'images').mkdir(parents=True, exist_ok=True)
    (base/'masks').mkdir(parents=True, exist_ok=True)
    for i in range(n):
        y = i % 2
        h = w = 160
        yy, xx = np.mgrid[:h, :w]
        brain = (((xx-w/2)/(w*.38))**2 + ((yy-h/2)/(h*.42))**2 < 1)
        img = rng.normal(0.35, 0.07, (h,w)).clip(0,1)
        img[brain] += 0.22
        mask = np.zeros((h,w), np.uint8)
        if y:
            cx, cy = rng.integers(w//2-20,w//2+20), rng.integers(h//2-20,h//2+20)
            lesion = ((xx-cx)**2/(rng.integers(7,14)**2) + (yy-cy)**2/(rng.integers(6,12)**2)) < 1
            mask[lesion & brain] = 255
            img[mask>0] += 0.18
        img = (np.clip(img,0,1)*255).astype(np.uint8)
        ip, mp = base/'images'/f'{i}.png', base/'masks'/f'{i}.png'
        Image.fromarray(img).save(ip); Image.fromarray(mask).save(mp)
        out.append({IMAGE_COL:str(ip), LABEL_COL:y, PATIENT_COL:f'patient_{i}',
                    MASK_COL:str(mp), LABEL_VALID_COL:1, MASK_VALID_COL:1})
    return pd.DataFrame(out)

if USE_SYNTHETIC_SMOKE_TEST:
    df = make_synthetic_frame()
else:
    if not Path(CSV_PATH).exists():
        raise FileNotFoundError(f'CSV not found: {CSV_PATH}. Set DATA_ROOT/CSV_PATH or enable smoke-test mode.')
    df = pd.read_csv(CSV_PATH)
    required = [IMAGE_COL, LABEL_COL, PATIENT_COL]
    missing = [c for c in required if c not in df.columns]
    if missing: raise ValueError(f'Missing required CSV columns: {missing}')
    df = df[df[IMAGE_COL].notna() & df[LABEL_COL].notna()].copy()
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    df = df[df[LABEL_COL].isin([0,1])].copy()
    # Verify files early
    bad = [p for p in df[IMAGE_COL].head(1000) if not resolve_path(p).exists()]
    if bad: raise FileNotFoundError(f'Example image path does not exist: {bad[0]}')

print('Rows:', len(df), '| labels:', df[LABEL_COL].value_counts().to_dict())
display(df.head())

In [ ]:
# Visualize sample CT-like image, multi-window channels, and mask
sample = df.iloc[min(1, len(df)-1)]
raw = read_gray(resolve_path(sample[IMAGE_COL]))
win = window_channels(raw)
fig, ax = plt.subplots(1, 5, figsize=(16,3))
ax[0].imshow(raw, cmap='gray'); ax[0].set_title('Input image')
for j, title in enumerate(['Window A','Window B','Window C']):
    ax[j+1].imshow(win[j], cmap='gray'); ax[j+1].set_title(title)
if MASK_COL in df.columns and resolve_path(sample.get(MASK_COL)) and resolve_path(sample.get(MASK_COL)).exists():
    ax[4].imshow(np.asarray(Image.open(resolve_path(sample[MASK_COL])).convert('L')), cmap='gray')
else: ax[4].text(.5,.5,'No mask',ha='center',va='center')
ax[4].set_title('Lesion mask')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

## 4. Patient-disjoint train/validation/calibration/test split
The split is performed by patient/group, never by slice alone. The calibration subset is reserved for temperature scaling and threshold selection. For small datasets, stratification may not be possible; inspect the distributions and adjust the ratios before interpreting results.

In [ ]:
def grouped_split(frame, group_col=PATIENT_COL, seed=SEED):
    # 70% train, 10% validation, 10% calibration, 10% test using patient groups.
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed)
    tr_idx, rem_idx = next(gss1.split(frame, frame[LABEL_COL], groups=frame[group_col]))
    tr = frame.iloc[tr_idx].copy()
    rem = frame.iloc[rem_idx].copy()
    # Split remaining 30% into three approximately equal partitions.
    gss2 = GroupShuffleSplit(n_splits=1, test_size=2/3, random_state=seed+1)
    va_idx, rem2_idx = next(gss2.split(rem, rem[LABEL_COL], groups=rem[group_col]))
    va = rem.iloc[va_idx].copy()
    rem2 = rem.iloc[rem2_idx].copy()
    gss3 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=seed+2)
    ca_idx, te_idx = next(gss3.split(rem2, rem2[LABEL_COL], groups=rem2[group_col]))
    ca, te = rem2.iloc[ca_idx].copy(), rem2.iloc[te_idx].copy()
    # Assert no patient leakage
    sets = [set(z[group_col].astype(str)) for z in (tr,va,ca,te)]
    assert all(not (sets[i] & sets[j]) for i in range(4) for j in range(i+1,4))
    return tr, va, ca, te

train_df, val_df, cal_df, test_df = grouped_split(df)
for name, part in [('train',train_df),('validation',val_df),('calibration',cal_df),('test',test_df)]:
    print(f'{name:12s}: n={len(part):4d}, labels={part[LABEL_COL].value_counts().to_dict()}')

## 5. AMAGT-LightNet / EETR-LightNet compact model
This runnable implementation approximates the manuscript's attention-modulated anatomy gate using a small CNN, attenuation statistics, coordinate/occupancy cues, a lightweight token-mixing block, a classification head, and a lesion decoder. Evidential outputs use non-negative evidence and Dirichlet strength. Topology loss is training-only. It is not the named pretrained TAP-CT 3D foundation encoder because those weights and architecture are not supplied.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1, groups=out_ch, bias=False),
            nn.Conv2d(out_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU())
    def forward(self,x): return self.net(x)

class ACUGLightNet(nn.Module):
    def __init__(self, num_classes=2, width=32):
        super().__init__()
        self.stem = ConvBlock(3, width, 2)
        self.enc2 = ConvBlock(width, width*2, 2)
        self.enc3 = ConvBlock(width*2, width*3, 2)
        d = width*3
        self.token_mix = nn.MultiheadAttention(d, num_heads=4, batch_first=True)
        self.norm = nn.LayerNorm(d)
        self.attn_stat = nn.Sequential(nn.Linear(6, d), nn.GELU(), nn.Linear(d, 2*d))
        self.anatomy_gate = nn.Sequential(nn.Linear(d+3, d), nn.Sigmoid())
        self.cls_head = nn.Sequential(nn.Linear(d, 128), nn.GELU(), nn.Dropout(.2),
                                      nn.Linear(128, num_classes))
        self.evidence_head = nn.Sequential(nn.Linear(d, 128), nn.GELU(),
                                           nn.Linear(128, num_classes), nn.Softplus())
        self.decoder = nn.Sequential(
            nn.Conv2d(d, width*2, 3, padding=1), nn.GELU(),
            nn.Conv2d(width*2, width, 3, padding=1), nn.GELU(),
            nn.Conv2d(width, 1, 1))
    def forward(self, x):
        # x [B,3,H,W]
        a = self.stem(x); b = self.enc2(a); z = self.enc3(b)
        B,C,H,W = z.shape
        tok = z.flatten(2).transpose(1,2)
        mixed,_ = self.token_mix(tok,tok,tok,need_weights=False)
        tok = self.norm(tok + mixed)
        pooled = tok.mean(1)
        # attenuation descriptors: channel means/stds from original window channels
        means = x.mean((2,3)); stds = x.std((2,3), unbiased=False)
        desc = torch.cat([means,stds],1)
        scale,shift = self.attn_stat(desc).chunk(2,dim=1)
        mod = pooled * (1 + 0.1*torch.tanh(scale)) + 0.1*shift
        coords = torch.tensor([0.5,0.5,1.0],device=x.device,dtype=x.dtype).view(1,3).expand(B,-1)
        gate = self.anatomy_gate(torch.cat([mod,coords],1))
        fused = gate*mod + (1-gate)*pooled
        logits = self.cls_head(fused)
        evidence = self.evidence_head(fused) + 1e-6
        lesion_logits = F.interpolate(self.decoder(z), size=x.shape[-2:], mode='bilinear', align_corners=False)
        return {'logits':logits, 'evidence':evidence, 'lesion_logits':lesion_logits,
                'embedding':fused}

model = ACUGLightNet().to(DEVICE)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 6. Losses and training
Classification uses only observed labels. Segmentation and topology terms are used only where an audited lesion mask is available. Topology regularization here is a differentiable morphological proxy (opening/closing mismatch plus out-of-mask penalty); it is not a formal guarantee of connectedness.

In [ ]:
def soft_dice_loss(logits, target, eps=1e-6):
    p = torch.sigmoid(logits)
    inter = (p*target).sum((1,2,3))
    denom = p.sum((1,2,3)) + target.sum((1,2,3))
    return (1 - (2*inter+eps)/(denom+eps)).mean()

def topology_proxy_loss(logits, target, mask_valid):
    p = torch.sigmoid(logits)
    # Penalize lesion probability outside the image-derived foreground proxy and small isolated noise.
    # In the absence of an explicit brain mask, use non-zero intensity support as a weak proxy.
    # This term is intentionally conservative; real CT projects should supply a brain mask.
    foreground = (target > 0).float()
    outside = (p * (1-foreground)).mean()
    # Morphological opening proxy
    eroded = -F.max_pool2d(-p, 3, stride=1, padding=1)
    opened = F.max_pool2d(eroded, 3, stride=1, padding=1)
    frag = (p-opened).abs().mean()
    return 0.05*frag + 0.02*outside

def evidential_loss(evidence, y, valid):
    # Expected categorical NLL under Dirichlet; invalid labels excluded.
    alpha = evidence + 1.0
    strength = alpha.sum(1, keepdim=True)
    probs = alpha / strength
    safe_y = y.clamp(min=0)
    nll = -torch.log(probs.gather(1,safe_y[:,None]).clamp_min(1e-8)).squeeze(1)
    return (nll*valid).sum() / valid.sum().clamp_min(1)

def batch_loss(out, y, masks, mask_valid, label_valid):
    valid_cls = label_valid * (y >= 0).float()
    ce = F.cross_entropy(out['logits'], y.clamp(min=0), reduction='none')
    cls = (ce*valid_cls).sum()/valid_cls.sum().clamp_min(1)
    evid = evidential_loss(out['evidence'], y, valid_cls)
    if mask_valid.sum() > 0:
        seg_each = F.binary_cross_entropy_with_logits(out['lesion_logits'], masks, reduction='none').mean((1,2,3))
        seg = (seg_each*mask_valid).sum()/mask_valid.sum().clamp_min(1) + soft_dice_loss(out['lesion_logits'][mask_valid.bool()], masks[mask_valid.bool()])
        topo = topology_proxy_loss(out['lesion_logits'], masks, mask_valid)
    else:
        seg = out['lesion_logits'].sum()*0
        topo = out['lesion_logits'].sum()*0
    return cls + 0.25*evid + 0.5*seg + 0.05*topo, {'cls':float(cls.detach()),'evid':float(evid.detach()),'seg':float(seg.detach()),'topo':float(topo.detach())}

def collate_ct(batch):
    xs,ys,ms,mv,lv,groups = zip(*batch)
    return torch.stack(xs), torch.stack(ys), torch.stack(ms), torch.stack(mv), torch.stack(lv), groups

train_loader = DataLoader(CT2DDataset(train_df, True), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=collate_ct)
val_loader = DataLoader(CT2DDataset(val_df), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, collate_fn=collate_ct)
cal_loader = DataLoader(CT2DDataset(cal_df), batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, collate_fn=collate_ct)
test_loader = DataLoader(CT2DDataset(test_df), batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, collate_fn=collate_ct)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type=='cuda'))
best_val, best_state, patience, bad_epochs = float('inf'), None, 4, 0
history = []
for epoch in range(EPOCHS):
    model.train(); tr_losses=[]
    for x,y,m,mv,lv,_ in train_loader:
        x,y,m,mv,lv = x.to(DEVICE),y.to(DEVICE),m.to(DEVICE),mv.to(DEVICE),lv.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type=='cuda')):
            out=model(x); loss, parts=batch_loss(out,y,m,mv,lv)
        scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer); scaler.update()
        tr_losses.append(loss.item())
    model.eval(); va_losses=[]
    with torch.no_grad():
        for x,y,m,mv,lv,_ in val_loader:
            x,y,m,mv,lv=x.to(DEVICE),y.to(DEVICE),m.to(DEVICE),mv.to(DEVICE),lv.to(DEVICE)
            out=model(x); loss,_=batch_loss(out,y,m,mv,lv); va_losses.append(loss.item())
    tr=float(np.mean(tr_losses)) if tr_losses else np.nan
    va=float(np.mean(va_losses)) if va_losses else tr
    history.append({'epoch':epoch+1,'train_loss':tr,'val_loss':va})
    print(f'Epoch {epoch+1:02d}/{EPOCHS} | train={tr:.4f} | val={va:.4f}')
    if va < best_val:
        best_val=va; best_state=copy.deepcopy(model.state_dict()); bad_epochs=0
    else:
        bad_epochs+=1
        if bad_epochs>=patience: print('Early stopping'); break
if best_state is not None: model.load_state_dict(best_state)
display(pd.DataFrame(history))

## 7. Validation, calibration, and test metrics
Temperature is fitted on the calibration partition only. The held-out test set is evaluated once after threshold selection. If a split contains only one class, AUROC is reported as NaN rather than fabricated.

In [ ]:
def predict_loader(model, loader):
    model.eval(); ys=[]; valid=[]; logits=[]; evid=[]; masks=[]; mask_logits=[]; groups=[]
    with torch.no_grad():
        for x,y,m,mv,lv,g in loader:
            x=x.to(DEVICE)
            out=model(x)
            ys.extend(y.numpy().tolist()); valid.extend(lv.numpy().tolist())
            logits.append(out['logits'].cpu()); evid.append(out['evidence'].cpu())
            masks.append(m.numpy()); mask_logits.append(out['lesion_logits'].cpu())
            groups.extend(list(g))
    return {'y':np.asarray(ys),'label_valid':np.asarray(valid).astype(bool),
            'logits':torch.cat(logits).numpy(),'evidence':torch.cat(evid).numpy(),
            'masks':np.concatenate(masks),'mask_logits':torch.cat(mask_logits).numpy(),'groups':groups}

val_pred = predict_loader(model,val_loader)
cal_pred = predict_loader(model,cal_loader)
test_pred = predict_loader(model,test_loader)

def fit_temperature(logits, y, valid):
    keep=valid & (y>=0)
    if keep.sum()<2: return 1.0
    lt=torch.tensor(logits[keep],dtype=torch.float32)
    yt=torch.tensor(y[keep],dtype=torch.long)
    log_t=nn.Parameter(torch.zeros(1))
    opt=torch.optim.LBFGS([log_t],lr=0.1,max_iter=50)
    def closure():
        opt.zero_grad()
        loss=F.cross_entropy(lt/torch.exp(log_t).clamp(0.05,20),yt)
        loss.backward(); return loss
    opt.step(closure)
    return float(torch.exp(log_t).clamp(0.05,20).detach())

temperature = fit_temperature(cal_pred['logits'],cal_pred['y'],cal_pred['label_valid'])
print('Calibration temperature:', round(temperature,4))

def softmax_np(z):
    z=z-z.max(axis=1,keepdims=True); e=np.exp(z); return e/e.sum(axis=1,keepdims=True)

cal_probs=softmax_np(cal_pred['logits']/temperature)
test_probs=softmax_np(test_pred['logits']/temperature)
# Select a confidence threshold on calibration data: target <=5% empirical selective risk when possible.
cy=cal_pred['y']; cv=cal_pred['label_valid'] & (cy>=0)
threshold=0.60
if cv.sum():
    best_cov=-1
    for th in np.arange(0.50,0.991,0.01):
        pred=cal_probs.argmax(1); accept=cal_probs.max(1)>=th
        sel=accept & cv
        if sel.sum() and (pred[sel]!=cy[sel]).mean() <= 0.05 and sel.mean()>best_cov:
            threshold=float(th); best_cov=sel.mean()
print('Selective confidence threshold:', round(threshold,3))

def expected_calibration_error(probs, y, valid, bins=10):
    keep=valid & (y>=0)
    if not keep.any(): return np.nan
    conf=probs.max(1)[keep]; pred=probs.argmax(1)[keep]; yy=y[keep]
    ece=0.
    for lo in np.linspace(0,1,bins+1)[:-1]:
        hi=lo+1/bins
        sel=(conf>=lo)&(conf<(hi if hi<1 else hi+1e-8))
        if sel.any(): ece += sel.mean()*abs((pred[sel]==yy[sel]).mean()-conf[sel].mean())
    return float(ece)

def classification_metrics(prediction, probs, threshold=None):
    y=prediction['y']; valid=prediction['label_valid'] & (y>=0)
    if valid.sum()==0: return pd.DataFrame([{'n_valid':0}])
    yt=y[valid]; pp=probs[valid]; yh=pp.argmax(1)
    row={'n_valid':int(valid.sum()),'Accuracy':accuracy_score(yt,yh),
         'Precision_macro':precision_score(yt,yh,average='macro',zero_division=0),
         'Recall_macro':recall_score(yt,yh,average='macro',zero_division=0),
         'Macro_F1':f1_score(yt,yh,average='macro',zero_division=0),
         'MCC':matthews_corrcoef(yt,yh),
         'Brier':float(np.mean(np.sum((pp-np.eye(pp.shape[1])[yt])**2,axis=1))),
         'ECE':expected_calibration_error(probs,y,valid)}
    try: row['AUROC'] = roc_auc_score(yt,pp[:,1]) if len(np.unique(yt))==2 else roc_auc_score(yt,pp,multi_class='ovr',average='macro')
    except Exception: row['AUROC']=np.nan
    try: row['AUPR'] = average_precision_score(yt,pp[:,1]) if len(np.unique(yt))==2 else np.nan
    except Exception: row['AUPR']=np.nan
    if threshold is not None:
        accept=pp.max(1)>=threshold
        row['Coverage']=float(accept.mean())
        row['Selective_risk']=float((yh[accept]!=yt[accept]).mean()) if accept.any() else np.nan
    return pd.DataFrame([row])

display(classification_metrics(test_pred,test_probs,threshold))
print(classification_report(test_pred['y'][test_pred['label_valid'] & (test_pred['y']>=0)],
                            test_probs.argmax(1)[test_pred['label_valid'] & (test_pred['y']>=0)],
                            zero_division=0))

## 8. Lesion localization metrics and uncertainty-guided refinement
Refinement is represented as a conservative local smoothing of high-entropy regions in this compact implementation. To reproduce the manuscript's DINO-Med3D crop refiner, provide that model's licensed weights and crop training code. Report segmentation metrics only for audited masks; no-mask cases are excluded.

In [ ]:
def dice_np(prob, target, thr=0.5, eps=1e-7):
    p=prob>=thr; t=target>=0.5
    return (2*np.logical_and(p,t).sum()+eps)/(p.sum()+t.sum()+eps)

def uncertainty_refine(prob, threshold_entropy=0.65):
    # prob [N,1,H,W]. Entropy is normalized to [0,1]; refine only high-entropy pixels.
    ent=-(prob*np.log(prob+1e-8)+(1-prob)*np.log(1-prob+1e-8))/np.log(2)
    high=ent>threshold_entropy
    refined=prob.copy()
    # local mean replacement only in uncertain areas
    import scipy.ndimage as ndi
    for i in range(len(prob)):
        local=ndi.uniform_filter(prob[i,0],size=3)
        refined[i,0][high[i,0]]=local[high[i,0]]
    return refined, ent

raw_mask_probs=1/(1+np.exp(-np.clip(test_pred['mask_logits'],-30,30)))
refined_probs, voxel_entropy=uncertainty_refine(raw_mask_probs)
seg_rows=[]
for i in range(len(raw_mask_probs)):
    if test_pred['masks'][i].sum()>0 or True:
        seg_rows.append({'sample':i,'dice_before':dice_np(raw_mask_probs[i,0],test_pred['masks'][i,0]),
                         'dice_after':dice_np(refined_probs[i,0],test_pred['masks'][i,0]),
                         'uncertain_fraction':float((voxel_entropy[i,0]>.65).mean())})
seg_df=pd.DataFrame(seg_rows)
display(seg_df.describe().T)

## 9. Review flag and abstention reasons
The review flag combines low maximum class confidence, high Dirichlet vacuity, and a large uncertain lesion area. Thresholds below are illustrative defaults; select them on the calibration subset only in a proper experiment. A review flag is not a clinical diagnosis.

In [ ]:
def review_signals(pred, probs, conf_th=threshold, vacuity_th=0.35, entropy_area_th=0.20):
    alpha=pred['evidence']+1
    vacuity=alpha.shape[1]/alpha.sum(1)
    mp=1/(1+np.exp(-np.clip(pred['mask_logits'],-30,30)))
    ent=-(mp*np.log(mp+1e-8)+(1-mp)*np.log(1-mp+1e-8))/np.log(2)
    uncertain_area=(ent>.65).mean(axis=(1,2,3))
    low_conf=probs.max(1)<conf_th
    high_vac=vacuity>vacuity_th
    high_spatial=uncertain_area>entropy_area_th
    review=low_conf|high_vac|high_spatial
    reasons=[]
    for a,b,c in zip(low_conf,high_vac,high_spatial):
        r=[]
        if a:r.append('low classification confidence')
        if b:r.append('high evidential vacuity')
        if c:r.append('high spatial entropy')
        reasons.append('; '.join(r) if r else 'no threshold trigger (not a clinical clearance)')
    return pd.DataFrame({'prediction':probs.argmax(1),'confidence':probs.max(1),
                         'vacuity':vacuity,'uncertain_area':uncertain_area,
                         'review_required':review,'reason':reasons})

review_df=review_signals(test_pred,test_probs)
display(review_df.head(20))
print('Review-required fraction:', round(review_df.review_required.mean(),3))

## 10. Guideline retrieval: BM25 + dense-like TF-IDF rank fusion with citation checks
Add legally obtained guideline passages to `guideline_passages.csv` with columns `passage_id`, `text`, `source`, and `url_or_page`. The fallback example below is a placeholder retrieval demonstration, not authoritative clinical guidance. For clinical use, replace it with exact, verified passages from the cited guideline editions, preserve page/section metadata, and require clinician review.

In [ ]:
GUIDELINE_CSV = f'{DATA_ROOT}/guideline_passages.csv'
if Path(GUIDELINE_CSV).exists():
    guideline_df=pd.read_csv(GUIDELINE_CSV)
    assert {'text','source'}.issubset(guideline_df.columns)
else:
    guideline_df=pd.DataFrame([
        {'passage_id':'placeholder-1','source':'PLACEHOLDER — replace with licensed AHA/ASA guideline passage',
         'text':'Acute stroke assessment requires interpretation of clinical findings and imaging by qualified clinicians.'},
        {'passage_id':'placeholder-2','source':'PLACEHOLDER — replace with licensed AHA/ASA guideline passage',
         'text':'Non-contrast CT is used in acute stroke pathways to assess intracranial hemorrhage and other abnormalities.'},
        {'passage_id':'placeholder-3','source':'PLACEHOLDER — replace with licensed AHA/ASA guideline passage',
         'text':'Imaging findings should be interpreted in context and uncertainty should prompt additional clinical review.'}
    ])
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
try:
    from rank_bm25 import BM25Okapi
    _has_bm25=True
except Exception:
    _has_bm25=False

texts=guideline_df['text'].fillna('').astype(str).tolist()
tokenized=[re.findall(r'\w+',t.lower()) for t in texts]
bm25=BM25Okapi(tokenized) if _has_bm25 else None
tfidf=TfidfVectorizer(ngram_range=(1,2),stop_words='english')
doc_mat=tfidf.fit_transform(texts) if texts else None

def retrieve_evidence(query, top_k=5):
    if not texts: return []
    qtok=re.findall(r'\w+',query.lower())
    if bm25 is not None: lex=np.asarray(bm25.get_scores(qtok))
    else: lex=np.asarray([sum(tok in t for tok in set(qtok)) for t in tokenized],dtype=float)
    qv=tfidf.transform([query]); dense=cosine_similarity(qv,doc_mat).ravel()
    lex_order=np.argsort(-lex)[:min(20,len(texts))]
    dense_order=np.argsort(-dense)[:min(20,len(texts))]
    fused={}
    for rank,idx in enumerate(lex_order,1): fused[idx]=fused.get(idx,0)+1/(60+rank)
    for rank,idx in enumerate(dense_order,1): fused[idx]=fused.get(idx,0)+1/(60+rank)
    order=sorted(fused,key=fused.get,reverse=True)[:top_k]
    return [{'index':i,'score':float(fused[i]),'text':texts[i],
             'source':str(guideline_df.iloc[i]['source']),
             'passage_id':str(guideline_df.iloc[i].get('passage_id',i))} for i in order]

def evidence_gate(row, evidence, similarity_floor=0.02):
    # Conservative gate: abstain on review flag, absent evidence, or placeholder source.
    if bool(row['review_required']): return False, 'model review flag triggered'
    if not evidence: return False, 'no retrieved evidence'
    if any('PLACEHOLDER' in e['source'].upper() for e in evidence):
        return False, 'guideline corpus contains placeholder passages'
    if evidence[0]['score'] < similarity_floor: return False, 'weak retrieval score'
    return True, 'retrieval passed basic checks; clinician verification still required'

for i in range(min(3,len(review_df))):
    row=review_df.iloc[i]
    query=f"CT stroke assessment predicted class {row.prediction}; confidence {row.confidence:.2f}; review required {row.review_required}"
    ev=retrieve_evidence(query,top_k=5)
    ok,reason=evidence_gate(row,ev)
    print(f'Case {i}:', 'CITED RESPONSE PERMITTED (not medical advice)' if ok else 'ABSTAIN')
    print('Reason:',reason)
    for e in ev[:2]: print(' -',e['source'], '|',e['passage_id'], '|',e['text'][:180])

## 11. Quantization and efficiency profiling
Dynamic INT8 quantization is a CPU-friendly approximation for eligible linear layers. It is not equivalent to the manuscript's TensorRT INT8 path. Compare accuracy/calibration before and after quantization and reject quantization if the predefined fidelity guard fails.

In [ ]:
def model_size_mb(m):
    import io
    buff=io.BytesIO(); torch.save(m.state_dict(),buff)
    return buff.tell()/1024**2

def profile_model(m, loader, device, max_batches=20):
    m.eval(); times=[]; n=0
    with torch.no_grad():
        for bi,batch in enumerate(loader):
            if bi>=max_batches: break
            x=batch[0].to(device)
            if device.type=='cuda': torch.cuda.synchronize()
            t0=time.perf_counter(); _=m(x)
            if device.type=='cuda': torch.cuda.synchronize()
            times.append((time.perf_counter()-t0)/len(x)); n+=len(x)
    return {'device':str(device),'n_profiled':n,'latency_ms_per_image':1000*np.mean(times) if times else np.nan,
            'model_state_mb':model_size_mb(m)}

fp32_profile=profile_model(model,test_loader,DEVICE)
print('FP model:',fp32_profile)
cpu_model=copy.deepcopy(model).to('cpu').eval()
try:
    int8_model=torch.ao.quantization.quantize_dynamic(cpu_model,{nn.Linear},dtype=torch.qint8)
    int8_profile=profile_model(int8_model,test_loader,torch.device('cpu'))
    print('Dynamic INT8 CPU model:',int8_profile)
    # Compare logits on a small CPU batch
    xb=next(iter(test_loader))[0][:min(8,BATCH_SIZE)]
    with torch.no_grad():
        p_fp=softmax_np(cpu_model(xb)['logits'].numpy())
        p_q=softmax_np(int8_model(xb)['logits'].numpy())
    print('Mean absolute probability difference:',float(np.mean(np.abs(p_fp-p_q))))
except Exception as e:
    int8_model=None
    print('Dynamic quantization unavailable in this runtime:',repr(e))

## 12. Ablation hooks and final report export
These switches let you define ablations in future experiments. A meaningful ablation requires retraining each variant with identical splits/seeds; simply zeroing a component at inference is not a substitute. The notebook avoids saving model checkpoints by default; it exports metric tables only.

In [ ]:
# Save experiment metrics (not model weights)
metrics = classification_metrics(test_pred,test_probs,threshold)
metrics['temperature'] = temperature
metrics['confidence_threshold'] = threshold
metrics['review_fraction'] = float(review_df.review_required.mean())
metrics['mean_uncertain_area'] = float(review_df.uncertain_area.mean())
metrics['mean_dice_before'] = float(seg_df.dice_before.mean())
metrics['mean_dice_after'] = float(seg_df.dice_after.mean())
metrics['device'] = str(DEVICE)
metrics['model_parameters'] = sum(p.numel() for p in model.parameters())
metrics['model_size_MB'] = model_size_mb(model)
display(metrics.T.rename(columns={0:'value'}))
metrics.to_csv('acug_lightnet_test_metrics.csv',index=False)
review_df.to_csv('acug_lightnet_review_flags.csv',index=False)
print('Wrote acug_lightnet_test_metrics.csv and acug_lightnet_review_flags.csv to the current notebook directory.')

## 13. Checklist before a real experiment

1. Turn `USE_SYNTHETIC_SMOKE_TEST=False` and verify all paths and patient IDs.
2. Use original calibrated HU data for actual HU windows; preserve orientation, spacing, and mask alignment.
3. Ensure label provenance is explicit. For each source, unobserved conditions must have `label_valid=0`, not label 0.
4. Do not mix slices from one patient across train/validation/calibration/test. For external evaluation, use independent sites with no threshold tuning.
5. Supply a valid brain mask and audited lesion masks to evaluate topology and segmentation properly.
6. Replace placeholder guideline passages with legally obtained, versioned source text and exact citations. Retrieval similarity alone is not clinical citation verification.
7. Report means and standard deviations across seeds (42, 2024, 3407), AUROC/AUPR, ECE/Brier, coverage/selective risk, Dice, latency, memory, and subgroup/external performance.
8. The notebook's 2D model and local refinement are engineering approximations. To claim exact ACUG-LightNet reproduction, implement/load the specified pretrained 3D TAP-CT and DINO-Med3D models and run the named datasets under the stated patient-level protocol.